In [4]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [5]:
len(documents)

72

In [6]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [7]:
# generating ground truth.
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [8]:
def user_prompt(doc):
  return {
    "role": "user",
    "content": f"Filename: {doc['filename']}\n\nPage: {doc['content']}"
  }

user_prompt(documents[0])


{'role': 'user',
 'content': 'Filename: 01-agentic-rag/lessons/01-intro.md\n\nPage: # Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word.

In [25]:
from evaluation_utils import llm_structured_retry
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
import json

class Questions(BaseModel):
  questions: list[str]

load_dotenv()
llm_client = OpenAI()

def format_questions(questions, doc):
  records = []
  for q in questions:
    records.append({
      'question': q,
      'document': doc['filename']
    })
  return records

def generate_ground_truth(doc):
  prompt = json.dumps(doc)
  result, usage = llm_structured_retry(llm_client, data_gen_instructions, prompt, Questions)
  return format_questions(result.questions, doc), usage
  

def doc_questions(doc, client, instructions, output_type):
  result, usage = llm_structured_retry(
    client=client,
    instructions=instructions,
    user_prompt=user_prompt(doc),
    output_type=output_type,
  )

  return {
    "filename": doc["filename"],
    "questions": result.questions,
    "usage": usage,
  }

  

In [10]:
# Q1: Generating questions

ground_truth = []

first_doc = next(doc for doc in documents if doc["filename"] == "01-agentic-rag/lessons/01-intro.md")
second_doc = next(doc for doc in documents if doc["filename"] == "01-agentic-rag/lessons/02-environment.md")
third_doc = next(doc for doc in documents if doc["filename"] == "01-agentic-rag/lessons/03-rag.md")

first_doc_result, first_doc_usage = generate_ground_truth(first_doc)
second_doc_result, second_doc_usage = generate_ground_truth(second_doc)
third_doc_result, third_doc_usage = generate_ground_truth(third_doc)






In [11]:
(first_doc_usage.input_tokens + second_doc_usage.input_tokens + third_doc_usage.input_tokens) / 3

1353.0

In [12]:
import pandas as pd

ground_truth = pd.read_csv("data/ground-truth.csv")
ground_truth.head()

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md


In [13]:
# from sentence_transformers import SentenceTransformer

# # 1. Downloads to the default Hugging Face cache
# model = SentenceTransformer('nvidia/Nemotron-3-Embed-1B-BF16') 

# # 2. Saves a permanent, structured copy to your specific directory
# model.save("../models/nemotron-3-embed-1b-bf16") 
 

In [27]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

from minsearch import Index, VectorSearch
import importlib
import embedder
importlib.reload(embedder)
from embedder import Embedder

index = Index(
  text_fields=["content"]
)
index.fit(chunks)

# Nemotron is asymmetric: index with "document" (passage: ), search with "query" (query: )
embedder = Embedder(path="../models/nemotron-3-embed-1b-bf16")
chunk_documents_embeddings = embedder.encode_batch(
    [chunk["content"] for chunk in chunks],
    prompt_name="document",
)

vector_search = VectorSearch()
vector_search.fit(chunk_documents_embeddings, chunks)


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'apply_yarn_scaling'}


In [28]:
def textSearch(query, num_results=5):
  return index.search(query, num_results=num_results)

def vecSearch(query, num_results=5):
  query_embedding = embedder.encode(query)
  return vector_search.search(query_embedding, num_results=num_results)

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60):
    text_results = textSearch(query, num_results=20)
    vector_results = vecSearch(query, num_results=20)
    return rrf([text_results, vector_results], k=k)

In [29]:
textSearch("What is RAG?")

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [30]:
vecSearch("What is RAG?")

[{'start': 0,
  'content': '# Retrieval Augmented Generation\n\nVideo: [RAG Workflows](https://youtu.be/FhGZV173xrk)\n\nAI Copilot solves the context problem for flow generation. But what about workflows that need to answer questions from your own data? That\'s where RAG comes in.\n\n> Note: Flows 1 and 2 use `{{ secret(\'GEMINI_API_KEY\') }}`. Flow 3 uses `{{ secret(\'OPENAI_API_KEY\') }}` and `{{ secret(\'TAVILY_API_KEY\') }}`. Make sure you\'ve completed the [setup instructions](03-setup.md) to configure the relevant secrets before running them.\n\n## What is RAG?\n\nRAG (Retrieval Augmented Generation) is a technique that retrieves relevant information from your data sources, augments the AI prompt with that context, and generates a response grounded in real data. This solves the hallucination problem by ensuring the AI has access to current, accurate information at query time.\n\nFor a deeper dive into RAG concepts, see [Module 1: Intro to RAG](../../01-agentic-rag/lessons/03-rag.

In [18]:
hybrid_search("What is RAG?")

[{'start': 7000,
  'content': ' use another judge to evaluate the\njudge. This is manual work, but it is necessary.\n\nA practical approach is to build a simple application with Streamlit.\nShow each question, the original answer, the generated answer, and the\njudge verdict side by side. Then mark each verdict as correct or\nincorrect and use that feedback to adjust the judge instructions. This\nis a lot of trial and error, but it makes the evaluation framework more\nreliable.\n\n## Saving the results\n\nSave the judged answers:\n\n```python\ndf_eval.to_csv("data/rag-evaluations-new.csv", index=False)\n```\n\nWe generated this file for the course materials on May 29, 2026. The\nrun used 395 RAG answers.\n\nThe results were:\n\n- Good: 379\n- Bad: 16\n\nThe total cost was $0.251331, about 25 cents.\n\nIf you don\'t want to run the judge yourself, download the file we\nprepared:\n\n```bash\nPREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main\nwget -O data/rag-evalua

In [19]:
#Q2. First result with text search
q = ground_truth.iloc[0]["question"]
textSearch(q)


[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [56]:
#Q3. First result with vector search
vecSearch(q)



[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [57]:
#Q4. First result with hybrid search
hybrid_search(q)

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [31]:
from tqdm.auto import tqdm

def compute_relevance(q, search_fn):
  filename = q["filename"]
  results = search_fn(q["question"])

  return [int(r["filename"] == filename) for r in results]

def compute_relavance_total(ground_truth, search_fn):
  relevance_total = []
  for q in tqdm(ground_truth):
    relevance_total.append(compute_relevance(q, search_fn))
  return relevance_total

def hit_rate(relevance_total):
  return sum(1 for r in relevance_total if any(r)) / len(relevance_total)

def reciprocal_rank(line):
  for rank, relevance in enumerate(line):
    if relevance:
      return 1 / (rank + 1)
  return 0

def mrr(relevance_total):
  return sum(reciprocal_rank(line) for line in relevance_total) / len(relevance_total)

def evaluate(ground_truth, search_fn):
  relevance_total = compute_relavance_total(ground_truth, search_fn)
  return {
    "hit_rate": hit_rate(relevance_total),
    "mrr": mrr(relevance_total)
  }

In [32]:
dict_ground_truth = ground_truth.to_dict(orient="records")
compute_relevance(dict_ground_truth[0], textSearch)

[0, 0, 0, 0, 1]

In [33]:
relevance_total = compute_relavance_total(dict_ground_truth, textSearch)

  0%|          | 0/360 [00:00<?, ?it/s]

In [120]:
hit_rate(relevance_total)

0.7583333333333333

In [121]:
mrr(relevance_total)

0.5942592592592593

In [122]:
evaluate(dict_ground_truth, textSearch)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5942592592592593}

In [22]:
evaluate(dict_ground_truth, vecSearch)

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.49166666666666664, 'mrr': 0.3609722222222222}

In [23]:
from functools import partial
evaluate(dict_ground_truth, partial(hybrid_search, k=60))

  0%|          | 0/360 [00:00<?, ?it/s]

{'hit_rate': 0.7583333333333333, 'mrr': 0.5657407407407408}

In [ ]:
for k in [1, 60]:
  print(k, evaluate(dict_ground_truth, partial(hybrid_search, k=k)))


  0%|          | 0/360 [00:00<?, ?it/s]

In [ ]:
# Baseline: Nemotron WITHOUT query/passage prefixes (before fix)
nemotron_no_prefix = {
    "text": {"hit_rate": 0.7583333333333333, "mrr": 0.5942592592592593},
    "vector": {"hit_rate": 0.49166666666666664, "mrr": 0.3609722222222222},
    "hybrid_k60": {"hit_rate": 0.7583333333333333, "mrr": 0.5657407407407408},
    "hybrid_k_sweep": {
        1: {"hit_rate": 0.7722222222222223, "mrr": 0.6039814814814815},
        50: {"hit_rate": 0.7583333333333333, "mrr": 0.5657407407407408},
        60: {"hit_rate": 0.7583333333333333, "mrr": 0.5657407407407408},
        100: {"hit_rate": 0.7583333333333333, "mrr": 0.5652777777777778},
        200: {"hit_rate": 0.7583333333333333, "mrr": 0.5652777777777778},
    },
}
nemotron_no_prefix

# After re-running cells above with prompt_name, store:
# nemotron_with_prefix = {
#   "text": ...,
#   "vector": ...,
#   "hybrid_k60": ...,
# }